# Synthetic light curves with PGMUVI

This tutorial uses the maintained helpers in `pgmuvi.synthetic` to create
deterministic one-dimensional and multiwavelength light curves with known
injected periods, amplitudes, phases, wavelength trends, and optional noise.

Every helper returns a `Lightcurve`, so the generated data can be passed into
the normal validation, period-diagnostic, fitting, and advisory workflows.


## What these generators do—and do not do

The functions used here evaluate analytic sinusoidal signals and then optionally
add Gaussian or Poisson-like observational noise. They are useful for controlled
examples and regression tests because the injected signal is known.

They do **not** draw a realization from a Gaussian-process prior or posterior.
GP-prior sampling requires an explicitly constructed model and kernel and remains
a separate advanced workflow. No GP model is created or trained in this notebook.


In [ ]:
from __future__ import annotations

import math

import numpy as np
import torch

from pgmuvi.dtypes import DEFAULT_DTYPE
from pgmuvi.synthetic import (
    make_chromatic_sinusoid_2d,
    make_multi_sinusoid_1d,
    make_multi_sinusoid_chromatic_2d,
    make_simple_sinusoid_1d,
)

SEED = 20260715

print(f"PGMUVI default dtype: {DEFAULT_DTYPE}")
print(f"synthetic seed: {SEED}")


## A reproducible single-period light curve

`make_simple_sinusoid_1d` creates a one-dimensional signal with a known period.
Irregular sampling and Poisson-like noise make the example less idealized while
the fixed seed keeps the result reproducible.


In [ ]:
simple_kwargs = {
    "n_obs": 120,
    "period": 90.0,
    "amplitude": 1.4,
    "phase": 0.25,
    "noise_level": 0.08,
    "noise_type": "poisson",
    "t_span": 360.0,
    "irregular": True,
    "seed": SEED,
}

lc_simple = make_simple_sinusoid_1d(**simple_kwargs)
lc_simple_repeat = make_simple_sinusoid_1d(**simple_kwargs)

assert lc_simple.ndim == 1
assert lc_simple.yerr is not None
assert torch.all(lc_simple.yerr > 0)
assert torch.allclose(lc_simple.xdata, lc_simple_repeat.xdata)
assert torch.allclose(lc_simple.ydata, lc_simple_repeat.ydata)

print(f"rows: {len(lc_simple.ydata)}")
print(f"input dimensions: {lc_simple.ndim}")
print(f"time span: {float(lc_simple.xdata.max() - lc_simple.xdata.min()):.3f}")
print(f"reproducible: {torch.allclose(lc_simple.ydata, lc_simple_repeat.ydata)}")


## Noise modes

The helpers support three noise choices:

- `"poisson"`: flux-dependent Poisson-like uncertainties;
- `"gaussian"`: constant standard deviation set by `noise_level`;
- `None`: no added noise; the returned `Lightcurve` has no `yerr` attribute.

A noise-free synthetic curve is useful for checking exact signal construction,
but realistic fitting examples should normally include strictly positive
uncertainties.


In [ ]:
lc_gaussian = make_simple_sinusoid_1d(
    n_obs=80,
    period=60.0,
    amplitude=1.0,
    noise_level=0.05,
    noise_type="gaussian",
    t_span=180.0,
    irregular=True,
    seed=SEED + 1,
)
lc_poisson = make_simple_sinusoid_1d(
    n_obs=80,
    period=60.0,
    amplitude=1.0,
    noise_level=0.05,
    noise_type="poisson",
    t_span=180.0,
    irregular=True,
    seed=SEED + 1,
)
lc_noise_free = make_simple_sinusoid_1d(
    n_obs=80,
    period=60.0,
    amplitude=1.0,
    noise_level=0.0,
    noise_type=None,
    t_span=180.0,
    irregular=False,
    seed=SEED + 1,
)

assert lc_gaussian.yerr is not None
assert lc_poisson.yerr is not None
assert getattr(lc_noise_free, "yerr", None) is None

print(f"Gaussian median uncertainty: {float(lc_gaussian.yerr.median()):.6f}")
print(
    "Poisson uncertainty range: "
    f"{float(lc_poisson.yerr.min()):.6f}–{float(lc_poisson.yerr.max()):.6f}"
)
print(
    "noise-free uncertainties absent: "
    f"{getattr(lc_noise_free, 'yerr', None) is None}"
)


## Multiple periods in one dimension

`make_multi_sinusoid_1d` sums explicit components. The component list is the
injected truth; it should be retained alongside any later period-recovery
experiment. Recovering every injected period is not guaranteed, because cadence,
baseline, noise, model choice, and optimization all matter.


In [ ]:
components_1d = [
    {"period": 120.0, "amplitude": 1.0, "phase": 0.0},
    {"period": 60.0, "amplitude": 0.35, "phase": math.pi / 3.0},
    {"period": 37.0, "amplitude": 0.20, "phase": math.pi / 2.0},
]

lc_multi_1d = make_multi_sinusoid_1d(
    n_obs=180,
    components=components_1d,
    noise_level=0.06,
    noise_type="gaussian",
    t_span=480.0,
    irregular=True,
    seed=SEED + 2,
)

injected_periods_1d = [component["period"] for component in components_1d]

assert lc_multi_1d.ndim == 1
assert lc_multi_1d.yerr is not None

print(f"rows: {len(lc_multi_1d.ydata)}")
print(f"injected periods: {injected_periods_1d}")


## A chromatic single-period light curve

`make_chromatic_sinusoid_2d` places time in the first input column and numeric
wavelength in the second. The example uses different observation counts per
band, a linear wavelength-amplitude law, and a linear phase law.

The robust central-95% amplitude summary below is descriptive only; it is not a
fitted wavelength model.


In [ ]:
wavelengths = [0.55, 0.80, 1.25]
expected_counts = [48, 60, 72]

lc_chromatic = make_chromatic_sinusoid_2d(
    n_per_band=expected_counts,
    period=150.0,
    amplitude=1.0,
    wavelengths=wavelengths,
    amplitude_law="linear",
    amplitude_slope=0.6,
    wl_ref=0.55,
    phase_law="linear",
    phase_slope=0.25,
    noise_level=0.05,
    noise_type="gaussian",
    t_span=600.0,
    irregular=True,
    seed=SEED + 3,
)

x_chromatic = lc_chromatic.xdata.detach().cpu().numpy()
y_chromatic = lc_chromatic.ydata.detach().cpu().numpy()
unique_wavelengths, actual_counts = np.unique(
    x_chromatic[:, 1],
    return_counts=True,
)

central95_amplitudes = {}
for wavelength in unique_wavelengths:
    band_flux = y_chromatic[x_chromatic[:, 1] == wavelength]
    q025, q975 = np.quantile(band_flux, [0.025, 0.975])
    central95_amplitudes[float(wavelength)] = float(q975 - q025)

assert lc_chromatic.ndim == 2
assert actual_counts.tolist() == expected_counts
assert lc_chromatic.yerr is not None

print(
    "rows per wavelength:",
    dict(zip(unique_wavelengths.tolist(), actual_counts.tolist(), strict=True)),
)
print("central-95% amplitudes:", central95_amplitudes)


## Extinction-law amplitude option

The chromatic generator also exposes the package's `"extinction"` amplitude
law. This is an analytic test signal controlled by `overall_amplitude`, `tau`,
`alpha`, and `offset`; using it does not establish that a physical dust model is
appropriate for an observed source.


In [ ]:
lc_extinction = make_chromatic_sinusoid_2d(
    n_per_band=40,
    period=220.0,
    wavelengths=[0.8, 1.2, 2.2],
    amplitude_law="extinction",
    overall_amplitude=5.0,
    tau=0.7,
    alpha=1.3,
    offset=0.2,
    phase_law="none",
    noise_level=0.04,
    noise_type="gaussian",
    t_span=660.0,
    irregular=True,
    seed=SEED + 4,
)

x_extinction = lc_extinction.xdata.detach().cpu().numpy()
extinction_wavelengths, extinction_count_values = np.unique(
    x_extinction[:, 1],
    return_counts=True,
)
extinction_counts = dict(
    zip(
        extinction_wavelengths.tolist(),
        extinction_count_values.tolist(),
        strict=True,
    )
)

print(f"input dimensions: {lc_extinction.ndim}")
print("rows per wavelength:", extinction_counts)


## Multi-period chromatic LPV-style data

`make_multi_sinusoid_chromatic_2d` combines multiple periods with
wavelength-dependent amplitudes and phases. The example uses a fundamental and
first harmonic and allows the number of observations to differ by band.

This is useful for testing diagnostics intended for multi-periodic data, but
current model-selection output remains advisory rather than automatic.


In [ ]:
components_2d = [
    {"period": 400.0, "amplitude_fraction": 0.40, "phase": 0.0},
    {
        "period": 200.0,
        "amplitude_fraction": 0.12,
        "phase": math.pi / 2.0,
    },
]

lc_multi_chromatic = make_multi_sinusoid_chromatic_2d(
    n_per_band=(45, 65),
    components=components_2d,
    wavelengths=[0.8, 1.2, 2.2],
    amplitude_law="extinction",
    overall_amplitude=5.0,
    tau=0.7,
    alpha=1.3,
    offset=0.2,
    phase_law="linear",
    phase_slope=0.08,
    noise_level=0.05,
    noise_type="gaussian",
    t_span=1200.0,
    irregular=True,
    seed=SEED + 5,
)

x_multi_chromatic = lc_multi_chromatic.xdata.detach().cpu().numpy()
multi_wavelengths, multi_counts = np.unique(
    x_multi_chromatic[:, 1],
    return_counts=True,
)
injected_periods_2d = [component["period"] for component in components_2d]

assert lc_multi_chromatic.ndim == 2
assert len(multi_wavelengths) == 3
assert all(45 <= int(count) <= 65 for count in multi_counts)

print(f"injected periods: {injected_periods_2d}")
print(
    "rows per wavelength:",
    dict(zip(multi_wavelengths.tolist(), multi_counts.tolist(), strict=True)),
)


## Prepare later analyses without running them here

Synthetic `Lightcurve` objects use the same interfaces as observed data. A
single-band mock can be passed to a `1D` fit, and a chromatic mock can be passed
to a baseline `2D` consensus fit or to explicit wavelength-dependent
comparisons.

The dictionaries below are examples only. This notebook deliberately does not
call `fit()`.


In [ ]:
prepared_fit_configs = {
    "single_period_1d": {
        "lightcurve": "lc_simple",
        "fit_kwargs": {
            "model": "1D",
            "training_iter": 100,
            "miniter": 20,
            "learn_additional_noise": True,
        },
    },
    "chromatic_baseline": {
        "lightcurve": "lc_chromatic",
        "fit_kwargs": {
            "model": "2D",
            "fit_strategy": "consensus",
            "training_iter": 100,
            "miniter": 20,
            "learn_additional_noise": True,
        },
    },
}

prepared_fit_configs


## Interpretation boundaries

A controlled synthetic experiment should record:

- generator name and all arguments;
- random seed;
- injected periods, amplitudes, phases, and wavelength law;
- sampling cadence and baseline;
- noise model and uncertainty scale; and
- the exact fit or advisory configuration used later.

Known injected truth makes recovery measurable, but it does not make every
configuration identifiable. Failure to recover a component may reflect the
sampling, noise, diagnostics, kernel family, constraints, or optimizer.


## Next steps

- Use the preprocessing tutorial to inspect sampling and variability before
  fitting: `notebooks/tutorial_preprocessing`.
- Use the 2-D tutorial for baseline consensus fitting:
  `notebooks/pgmuvi_tutorial_2d`.
- Use `howto/wavelength_models` and `howto/wavelength_advisory` for explicit
  wavelength-model comparisons.
- Consult the `pgmuvi.synthetic` API reference for every generator argument.
- GP-prior and posterior-predictive sampling remain a separate pending notebook
  workflow; do not infer that the analytic helpers perform those operations.
